[Reference](https://medium.com/data-science-collective/building-llm-memory-from-scratch-2-auto-summarization-buffers-2e2cba08c4ca)

# Step 0: Decide Invariants

In [1]:
CONTEXT_WINDOW = 4096 # Model context length in tokens
SUMMARY_TOKEN_BUDGET = 1024 # Max tokens for summary generation
GENERATION_MAX_TOKENS = 1024 # Reserved for the model's reply
SAFETY_MARGIN = 32 # Stop tokens, wiggle room

In [2]:
SYSTEM_PROMPT = (
"You are a friendly and knowledgeable AI assistant. "
"IMPORTANT: You have access to a summary of your previous conversation with the user. "
"This summary contains facts you learned about the user (like their name, preferences, etc.) "
"and topics you discussed. When answering questions, USE THIS SUMMARY as if it is your memory. "
"If the user asks about something mentioned in the summary, answer confidently based on that information. "
"Speak in first person ('I') and address the user directly ('you')."
)

# Step 1: Spin Up Llama-cpp Instance

In [6]:
!pip install llama-cpp-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 MB 14.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.8 MB/s eta 0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.16-cp312-cp312-linux_x86_64.whl size=4422318 sha256=151bbcfb72f14c1a6dd5ad6b027e3342c5c2cefd16fa8517519c6103a5ded9fd
  Stored in directory: /root/.cache/pip/wheels/90/82/ab/8784ee3fb99ddb07fd36a679ddbe63122cc07718f6c1eb3be8
Successfully built llama-cpp-python


In [8]:
import llama_cpp

llm = llama_cpp.Llama(
 model_path="./models/qwen2.5-7b-instruct-q5_k_m-00001-of-00002.gguf",
 n_ctx=CONTEXT_WINDOW,
 n_gpu_layers=-1,
 verbose=False,
)

In [9]:
import llama_cpp

class LLMWithAutoSummarization:
    """
    Chat wrapper around llama-cpp that keeps a single running summary
    of the whole conversation. The raw dialogue is **never** sent back
    to the model once it has been summarized.
    """

    SYSTEM_PROMPT = (
        "You are a friendly and knowledgeable AI assistant. "
        "IMPORTANT: You have access to a summary of your previous conversation with the user. "
        "This summary contains facts you learned about the user (like their name, preferences, etc.) "
        "and topics you discussed. When answering questions, USE THIS SUMMARY as if it is your memory. "
        "If the user asks about something mentioned in the summary, answer confidently based on that information. "
        "Speak in first person ('I') and address the user directly ('you')."
    )

    CONTEXT_WINDOW = 10240
    SUMMARY_TOKEN_BUDGET = 1024
    GENERATION_MAX_TOKENS = 2048

    def __init__(self, model_path: str = "./models/qwen2.5-7b-instruct-q5_k_m-00001-of-00002.gguf"):
        self.llm = llama_cpp.Llama(
            model_path=model_path,
            n_gpu_layers=-1,
            n_ctx=self.CONTEXT_WINDOW,
            verbose=False,
        )
        self.summary: str = ""
        self.generation_params = {
            "temperature": 0.7,
            "top_p": 0.9,
            "stop": ["<|im_end|>", "<|endoftext|>"],
            "max_tokens": self.GENERATION_MAX_TOKENS,
        }

    def answer(self, user_text: str) -> str:
        """Generate a reply and immediately update the running summary."""
        # --- Phase 1: Answer ---
        prompt_msgs = [{"role": "system", "content": self.SYSTEM_PROMPT}]

        if self.summary:
            prompt_msgs.append({
                "role": "system",
                "content": (
                    "HERE IS WHAT YOU KNOW FROM YOUR PREVIOUS CONVERSATION:\n\n"
                    f"{self.summary}\n\n"
                    "Use this information to answer the user's questions. "
                    "If they ask about their name, location, preferences, or anything else "
                    "mentioned above, provide that information confidently."
                )
            })

        prompt_msgs.append({"role": "user", "content": user_text})

        reply = self.llm.create_chat_completion(
            messages=prompt_msgs,
            **self.generation_params
        )["choices"][0]["message"]["content"]

        # --- Phase 2: Update memory ---
        self._update_summary(user_text, reply)

        return reply

    def _update_summary(self, user_text: str, assistant_text: str) -> None:
        """Roll the latest exchange into the running summary."""
        # (Insert the complete _update_summary code from Step 5)
        ...

    def print_memory(self) -> None:
        """Inspect the current summary."""
        print("\n=== Running Summary ===")
        print(self.summary or "(empty)")
        print("=======================\n")

In [10]:
llm_with_memory = LLMWithAutoSummarization()
print(llm_with_memory.answer("My name is Alice.")) # Turn 1
# Simulate 10 more turns of conversation about something else
for i in range(10):
 print(f"--- Turn {i+2} ---")
 print(llm_with_memory.answer(f"Tell me a random fact about the number {i}."))
 # Now, ask the critical question again
 print("--- The Moment of Truth ---")
 print(llm_with_memory.answer("What is my name?"))

In [11]:
llm_with_memory = LLMWithAutoSummarization()
print(llm_with_memory.answer("My name is Alice.")) # Turn 1
# Simulate 10 more turns of conversation about something else
for i in range(10):
 print(f"--- Turn {i+2} ---")
 print(llm_with_memory.answer(f"Tell me a random fact about the number {i}."))
 # Now, ask the critical question again
 print("--- The Moment of Truth ---")
 print(llm_with_memory.answer("What is my name?"))